# 02 — BioKG training, evaluation and checkpoint verification

This notebook replaces the duplicate EDA draft with the corrected model workflow. It calls
repository modules rather than maintaining a second model implementation. It contains no saved
performance claims and never assigns random explanation importance.

The encoder uses identity embeddings for the observed BioKG entities. It does not support
unseen-drug generalization. DistMult is symmetric for a same-type pair at a fixed relation;
it cannot represent arbitrary directional effects. A sigmoid score is **uncalibrated**, not a
clinical probability. Complete evaluation is a serving prerequisite, not proof of model quality.


## 1. Setup and official training graph

Use the same Python 3.11 environment as [Notebook 01](01_ogbl_biokg_eda.ipynb). Data loading is
explicit. Training, test evaluation and scoring remain separately disabled until you enable them.
No training is performed during web startup.


In [ ]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
import torch
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/data/load_biokg.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook from the repository root or its notebooks folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data.load_biokg import load_dataset, load_biokg_as_hetero
from src.utils.graph_contract import graph_contract, flatten_graph, query_indices

# Set BIOKG_NOTEBOOK_RUN_DATA=1 only when official data access/disk are available.
# Default execution verifies setup and skips data-dependent cells explicitly.
RUN_DATA = os.getenv("BIOKG_NOTEBOOK_RUN_DATA", "0") == "1"
dataset = graph = contract = None
print({"python": sys.version.split()[0], "torch": torch.__version__,
       "pandas": pd.__version__, "official_data_enabled": RUN_DATA})


In [ ]:
if RUN_DATA:
    dataset = load_dataset()  # Downloads official OGB data only if not cached.
    graph = load_biokg_as_hetero(dataset)  # Uses dataset.load_split("train") only.
    contract = graph_contract(graph)
    print({"train_graph_sha256": contract["train_graph_sha256"],
           "node_types": contract["node_types"], "relations": len(contract["relations"])})
else:
    print("SKIPPED: official BioKG data is disabled; no substitute data is used.")


## 2. Shared indexing and decoder contract

The homogeneous graph adds the saved entity-type offset to each local head/tail index.
The selected relation indexes a learned DistMult embedding. Scoring is `sum(h * relation * t)`;
ignoring the relation or reusing legacy indices changes the model contract.


In [ ]:
if contract is not None:
    display(pd.DataFrame([{"entity_type": kind, "offset": contract["offsets"][kind],
                           "nodes": contract["num_nodes"][kind]}
                          for kind in contract["node_types"]]))
    drug_relations = [rel for src, rel, dst in contract["relations"] if src == dst == "drug"]
    if not drug_relations:
        raise RuntimeError("Training graph contains no drug-drug relation")
    QUERY_RELATION = drug_relations[0]
    print({"query_local_indices": [58, 89], "recorded_relation_id": QUERY_RELATION,
           "model_indices": query_indices(contract, 58, QUERY_RELATION, 89)})
else:
    print("SKIPPED: contract and query mapping require official data.")


## 3. Train candidates and select with validation only

Choose the budget, architecture and seed using validation. `steps` counts optimizer updates,
not epochs. The example budget below is a starting configuration, not a recommended or validated
recipe. Full-graph R-GCN training may need substantial memory. Training negatives are type-correct
corruptions excluding known training positives; unknown positives can still be sampled.

A smoke run with `--max-eval` verifies execution only and produces an incomplete validation report.
It cannot enable serving or proceed to final test evaluation. Never report smoke MRR as BioKG
performance. Use distinct candidate paths to preserve experiments.


In [ ]:
import subprocess
RUN_TRAINING = False
CANDIDATE = ROOT / "models/rgcn_candidate.pt"
RECIPE = {"steps": 100, "batch-size": 1024, "hidden-channels": 64,
          "learning-rate": 0.001, "seed": 42, "device": "cpu"}
train_command = [sys.executable, "-m", "src.train.train_rgcn", "--output", str(CANDIDATE)]
for key, value in RECIPE.items():
    train_command.extend(["--" + key, str(value)])
print("Candidate training command:", train_command)
if RUN_TRAINING:
    if not RUN_DATA:
        raise RuntimeError("Enable official data first")
    if CANDIDATE.exists():
        raise FileExistsError("Choose a new candidate path to preserve earlier weights")
    subprocess.run(train_command, cwd=ROOT, check=True)
else:
    print("SKIPPED: training is disabled; no optimization or checkpoint creation occurred.")


In [ ]:
from src.utils.checkpoint import load_checkpoint
from src.train.evaluate_checkpoint import require_selected_candidate
if contract is not None and CANDIDATE.is_file():
    candidate_model, candidate = load_checkpoint(CANDIDATE, contract, "cpu", require_evaluation=False)
    print(json.dumps({"optimizer_steps": candidate["optimizer_steps"],
                      "training_config": candidate.get("training_config"),
                      "validation": candidate.get("evaluation", {}).get("valid")}, indent=2))
    del candidate_model
    # This gate rejects partial validation and candidates already exposed to test.
    require_selected_candidate(candidate)
else:
    print("SKIPPED: a trained compatible candidate and official graph are required.")


## 4. Freeze the recipe, then evaluate the selected weights on test

First compare complete validation results, choose one candidate and record the decision below.
The finalizer preserves that checkpoint's weights, performs an exact train/held-out separation
audit and evaluates the full official test split with OGB-provided typed negative candidates.
MRR uses both head and tail ranking and OGB tie handling. The finalizer has no optimizer.

Do not tune the recipe from test results or repeatedly select candidates by test MRR. The command
writes a new serving artifact and provenance report; it does not overwrite the candidate. A failed
separation audit must be investigated, not bypassed.


In [ ]:
RUN_TEST_EVALUATION = False
SELECTION_NOTE = ""  # Record validation comparison and the frozen recipe before test evaluation.
SERVING_CHECKPOINT = ROOT / "models/rgcn_v2.pt"
if RUN_TEST_EVALUATION:
    if not RUN_DATA or not CANDIDATE.is_file() or not SELECTION_NOTE.strip():
        raise RuntimeError("Official data, selected trained weights and a validation selection note are required")
    command = [sys.executable, "-m", "src.train.evaluate_checkpoint",
               "--checkpoint", str(CANDIDATE), "--output", str(SERVING_CHECKPOINT),
               "--selection-note", SELECTION_NOTE, "--device", "cpu"]
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("SKIPPED: final test evaluation is disabled; no test performance is reported.")


## 5. Load and score only a compatible fully evaluated checkpoint

Serving calls the same strict loader below. Legacy architecture, changed graph indexing, absent
optimization, or incomplete validation/test reports are rejected. Keep `ENABLE_PREDICTIONS=false`
when such an artifact is unavailable. A successful compatibility gate establishes neither
calibration nor clinical validity. Recorded graph neighborhoods remain available independently.


In [ ]:
RUN_SCORING = False
if RUN_SCORING:
    if graph is None or not SERVING_CHECKPOINT.is_file():
        raise RuntimeError("Official graph and a fully evaluated v2 checkpoint are required")
    model, checkpoint = load_checkpoint(SERVING_CHECKPOINT, contract, "cpu")
    edges, edge_types = flatten_graph(graph, contract, "cpu")
    head, relation, tail = query_indices(contract, 58, QUERY_RELATION, 89)
    with torch.inference_mode():
        embeddings = model(graph.x_dict, edges, edge_types)
        score = model.score(embeddings, torch.tensor(head), torch.tensor(relation),
                            torch.tensor(tail)).sigmoid().item()
    if not 0 <= score <= 1:
        raise RuntimeError("Nonfinite score")
    print({"drug1_local_index": 58, "drug2_local_index": 89,
           "relation_id": QUERY_RELATION, "Uncalibrated model score": score})
else:
    print("SKIPPED: model scoring requires an evaluated artifact and explicit opt-in.")


## 6. Evidence and implementation boundaries

| Item | What it establishes |
| --- | --- |
| Recorded training edge | A relationship represented in the source dataset |
| Model score | Experimental relation-specific model output for observed entities |
| Official validation/test MRR | Ranking under the provided typed candidate protocol |
| Synthetic regression tests | Indexing, scoring, optimization, loading and route mechanics |
| Clinical harm or treatment evidence | Not established by this demo |

The random transductive split can contain related or reversed facts across partitions. Exact
triple exclusion does not establish prospective or unseen-entity performance. Identity embeddings,
DistMult symmetry, incomplete/noisy source data and possibly false training negatives limit
interpretation. Calibration, clinical validation and meaningful attribution remain separate work.
The legacy explanation endpoints remain disabled. DrugBank adds optional metadata only and is
not a replacement model training graph.

Data-dependent cells require the official dataset. Training requires a new v2 candidate; final
test evaluation requires full validation plus a fixed selection decision; scoring requires a
compatible complete evaluation artifact. See [the README](../README.md) for deployment and checks.


## 7. Compare foundation checkpoints and description features

The matched entity comparison uses the same validation rows and official typed negatives for TRIX, ULTRA and R-GCN. The relation comparison ranks supported BioKG relation types for a fixed drug pair. Read the query counts and completion flags before comparing MRR.

[Evaluation protocol and reproduction commands](../docs/biokg-model-evaluation.md).


In [ ]:
import json

summary = json.loads((ROOT / "docs" / "biokg-results" / "validation-summary.json").read_text())
display(pd.DataFrame(summary["runs"]))
display(pd.DataFrame({
    name: values for name, values in summary["description_ablation"].items()
    if name != "description_coverage"
}).T)
